<h1><font style="color:blue">Project 2: Kaggle Competition - Classification</font> </h1>

In [1]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:98% !important; }</style>"))
%load_ext autoreload  
%autoreload 2
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import os
import numpy as np
import lightning as L
import torch.utils.data as data
from pathlib import Path
from dataclasses import dataclass
# from lightning.pytorch.demos.boring_classes import RandomDataset
# import zipfile
# import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms.functional as TVF
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
 
from torchinfo import summary
from sklearn.model_selection import train_test_split


# Use torchmetrics API to track and compute metrics automatically
# import our library
import torchmetrics.functional as TMF
from torchmetrics import MeanMetric    # Aggregate a stream of value into their mean value.
from torchmetrics.classification import MulticlassAccuracy # Compute Accuracy for multiclass tasks.

from PIL import Image
pd.set_option('display.width',180)

In [ ]:
import sys 
import gc
for p in ['../..']:
    if p not in sys.path:
        # print(f"insert {p}")
        sys.path.insert(0, p)
print(sys.path)

from training_code import *
from types import SimpleNamespace

In [4]:
# Plot few images
def display_images(imgs, lbls = None, names=None):
    plt.rcParams["figure.figsize"] = (18, 9)
    plt.figure
    if names is None:
        names = ['']*len(imgs)
    if lbls is None:
        lbls = ['']*len(imgs)
    for i in range(len(imgs)):
        plt.subplot(3, 6, i+1)
        # print(f" {type(imgs[i])}  {imgs[i].min():5f}  {imgs[i].max():5f}")
        img = TVF.to_pil_image(imgs[i])
        plt.imshow(img)
        plt.gca().set_title('Tgt: {0} {1}'.format(lbls[i], names[i]), fontsize=9)
    plt.show()

def display_image(img, lbl = None):
    # plt.rcParams["figure.figsize"] = (15, 9)
    plt.figure(figsize=(4,4))
    lbl = '' if lbl is None else lbl
    if img.ndim == 4 :
        img = img.squeeze()
        print(img.shape)
    img = TVF.to_pil_image(img) if isinstance(img, torch.Tensor) else img
    plt.imshow(img)
    plt.gca().set_title('Target: {0} '.format(lbl))
    plt.show()
        # break

def print_model_summary(model = None, batch_size = 4, image_size =  (3, 224, 224), depth = 3):
    summary_col_names = ("input_size",
                "output_size",
                "num_params",
                # "params_percent",
                "kernel_size",
                "mult_adds",
                "trainable" )

    input_size = [batch_size, *image_size]
    print(f"\nInput size: {input_size}")
    print(summary(model, input_size = input_size, col_names=summary_col_names, depth =depth))

# print(f"\n {model.name}")
# print(summary(model, input_size = input_size, col_names=summary_col_names))

mean = torch.tensor([0.485, 0.456, 0.406])
std  = torch.tensor([0.229, 0.224, 0.225])

def denormalize(y, mean, std) :
    match y.ndim:
        case 3:
            return y * std[:,None,None] + mean[:,None,None]
        case 4:
            return y * std[None,:,None,None] + mean[None, :,None,None]


def delete_vars(variables = ['model',  'metrics', 'train_config', 'sys_config', 'optimizer', 'scheduler', 'data_module']):
    for vr in variables:
        global_vars = globals()
        if vr in global_vars:
            try:
                del globals()[vr]
                print(f" {vr:15s} deleted . . . ")
            except Exception as e:
                print(e)
        else:
            print(f" {vr:15s} NOT defined . . .")
    
    print(f" gc.collect() : {gc.collect()}")
    print(f" Cuda empty cache : {torch.cuda.empty_cache()}")

# <font style="color:green">Helper Code </font>

## Date and time

In [1]:
from datetime import datetime
import time

In [2]:
now = datetime.now()

current_time = now.strftime("%d%m_%H:%M:%S")
print("Current Time =", current_time)

Current Time = 2002_12:53:00


## Arg Parser

In [ ]:
import argparse
def parse_args(input = None):
    parser = argparse.ArgumentParser(description="DNN classifier with SNNL")
    grp = parser.add_argument_group("Parameters")
    grp.add_argument("--config","--configuration", type=str, dest="configuration", required=True, help=" yaml file containing hyperparameters to use")
    grp.add_argument('--ckpt', type=str, required=False, default=None, help="Checkpoint fle to resume training from")
    grp.add_argument('--cpb', type=int, required=True, default=0, help="Compounds per batch" )    
    grp.add_argument('--exp_title',type=str, required=False, default=None, help="Exp Title (overwrites yaml file value)")
    grp.add_argument('--epochs', type=int, required=True, default=0, help="epochs to run")
    grp.add_argument('--gpu_id', type=int, required=False, default=0, help="Cuda device id to use" )    
    grp.add_argument('--lr', type=float, required=False, default=None, dest='learning_rate', help="Learning Rate" )    
    grp.add_argument('--run_id', type=str, required=False, default=None, dest='exp_id',  help="WandB run id (for run continuations)")
    grp.add_argument("--runmode", type=str, required=False, choices=['baseline', 'snnl'], default="base",
                     help="the model running mode: [baseline (default) | snnl]")
    grp.add_argument("--seed", type=int, required=False, default=1234, dest='random_seed', help="the random seed value to use, default: [1234]")
    grp.add_argument('--prim_opt', default=True, dest="use_prim_optimizer", action=argparse.BooleanOptionalAction)
    grp.add_argument('--temp_opt', default=False, dest="use_temp_optimizer", action=argparse.BooleanOptionalAction)
    grp.add_argument('--temp_annealing', default=False, dest="use_annealing", action=argparse.BooleanOptionalAction)
    grp.add_argument('--anneal_patience', type=int, required=False, default=15, dest="anneal_patience", help="annealing patience")
    grp.add_argument('--single_loss', default=False, dest="use_single_loss", action=argparse.BooleanOptionalAction,
                     help="Optimize Primary and SNNL loss together (or seperately when = False)")
    grp.add_argument('--temp', type=float, required=False, default=None, dest='temperature', help="Temperature initial value" )
    grp.add_argument('--adam_wd', type=float, required=False, default=None, dest='adam_weight_decay', help="Temperature initial value" )
    grp.add_argument('--loss_factor', type=float, required=False, default=None, dest='loss_factor', help="Primary Loss Factor" )
    grp.add_argument('--snnl_factor', type=float, required=False, default=None, dest='snnl_factor', help="SNN Loss Factor " )
    grp.add_argument('--temp_lr', type=float, required=False, default=None, dest='temperatureLR', help="Temperature learning rate")
    grp.add_argument('--wandb', default=False, required=False, dest='WANDB_ACTIVE', action=argparse.BooleanOptionalAction)
    arguments = parser.parse_args(input)
    return arguments

In [ ]:
# if __name__ == "__main__":
cli_args = f" --seed                1234 " \
             f" --runmode             baseline" \
             f" --configuration       ./hyperparameters/ae_cp_250_512_cpb.yaml" \
             f" --gpu_id              2 "\
             f" --cpb                 200" \
            f" --epochs              20"
            # f" --runmode           snnl" \
            # f" --configuration     hyperparameters/dnn_mnist.json"
print(f" {cli_args.split()}")
cli_args = parse_args(cli_args.split())
cli_args



In [ ]:
print(f" {cli_args}")

## toml file

In [5]:
import tomllib

with open("settings_1.toml", "rb") as f:
    data = tomllib.load(f)
data

{'BATCH_SIZE': 96,
 'EPOCHS': 200,
 'LEARNING_RATE': 0.01,
 'NUM_WORKERS': 6,
 'TUNING_LAYERS': ['fc', 'layer4', 'layer3'],
 'PROJECT_NAME': 'OpenCV_Pytorch_Week7',
 'DROPOUT_RATE': 0.0,
 'WEIGHT_DECAY': 0.0075,
 'LAST_EPOCH': -1,
 'LR_FACTOR': 0.3,
 'AUGMENTATION': 'V5',
 'WARMUP_ITERS': 10,
 'SCHEDULER': 'Cosine',
 'OPTIMIZER': 'SGD',
 'CLASS_BOOST': [1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1]}

In [13]:
train_config = TrainingConfiguration()
train_config.__dict__

{'augmentation': '',
 'batch_size': 4,
 'class_boost': None,
 'data_root': './images',
 'device': 'cuda',
 'dropout_rate': 0.0,
 'early_stopping': False,
 'epochs': 20,
 'image_shape': (224,),
 'learning_rate': 0.001,
 'last_epoch': -1,
 'log_interval': 5,
 'logs_root': './logs',
 'lr_factor': 1.0,
 'model_name': 'unknown',
 'notebook_name': '',
 'num_workers': 2,
 'optimizer': '',
 'project_name': '',
 'run_name': '',
 'session_name': '',
 'scheduler': '',
 'subset_size': 1.0,
 'test_interval': 1,
 'tuning_layers': '',
 'warmup_iters': 0,
 'weight_decay': 0.0001}

In [ ]:
for k,v in data.items():
    k = k.lower()
    # print(f" {k:25s}  {type(v)}  {v:}    {data.get(k, v)}")
    if k not in train_config.__dict__.keys():
        print(f" '{k}'  not found in training configuration - added . . . ")
 
    setattr(train_config, k, v)

In [16]:
train_config.__dict__
#  for k,v in train_config.__dict__.items():
#     print(f" {k:25s}  {type(v)}  {v:}") 

{'augmentation': 'V5',
 'batch_size': 96,
 'class_boost': [1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1],
 'data_root': './images',
 'device': 'cuda',
 'dropout_rate': 0.0,
 'early_stopping': False,
 'epochs': 200,
 'image_shape': (224,),
 'learning_rate': 0.01,
 'last_epoch': -1,
 'log_interval': 5,
 'logs_root': './logs',
 'lr_factor': 0.3,
 'model_name': 'unknown',
 'notebook_name': '',
 'num_workers': 6,
 'optimizer': 'SGD',
 'project_name': 'OpenCV_Pytorch_Week7',
 'run_name': '',
 'session_name': '',
 'scheduler': 'Cosine',
 'subset_size': 1.0,
 'test_interval': 1,
 'tuning_layers': ['fc', 'layer4', 'layer3'],
 'warmup_iters': 10,
 'weight_decay': 0.0075}

## Torchmetrics

In [ ]:
preds = torch.randn(10, 5).softmax(dim=-1)
target = torch.randint(5, (10,))
print(preds)
print(target)

In [ ]:
acc = TMF.accuracy(preds, target, task="multiclass", num_classes=5)
print(acc)

### Multiclass AOC

In [ ]:
from torch import tensor
from torchmetrics.classification import MulticlassAUROC
preds = tensor([[0.75, 0.05, 0.05, 0.05, 0.05],
                [0.05, 0.75, 0.05, 0.05, 0.05],
                [0.05, 0.05, 0.75, 0.05, 0.05],
                [0.05, 0.05, 0.05, 0.75, 0.05]])
target = tensor([0, 1, 3, 2])

In [ ]:
metric = MulticlassAUROC(num_classes=5, average="macro", thresholds=None)
metric(preds, target)

In [ ]:
mc_auroc = MulticlassAUROC(num_classes=5, average=None, thresholds=None)
mc_auroc(preds, target)

In [ ]:
mc_auroc = MulticlassAUROC(num_classes=5, average="macro", thresholds=5)
mc_auroc(preds, target)

In [ ]:
mc_auroc = MulticlassAUROC(num_classes=5, average=None, thresholds=5)
mc_auroc(preds, target)

In [ ]:
metric = MulticlassAUROC(num_classes=3, average=None)
values = [ ]
for _ in range(10):
    mc_auroc = metric(torch.randn(20, 3), torch.randint(3, (20,)))
    print(mc_auroc)
    values.append(mc_auroc)
    fig_, ax_ = metric.plot(values)
    plt.show()

In [ ]:
torch.randn(20, 13).shape, torch.randint(13, (20,)).shape

In [ ]:
metric = MulticlassAUROC(num_classes=13, average="macro", thresholds=10)
values = [ ]
for i in range (10):
    for _ in range(100):
        mc_auroc = metric(torch.randn(20, 13), torch.randint(13, (20,)))
    print(mc_auroc)
    values.append(mc_auroc)
        
fig_, ax_ = metric.plot(values)
plt.show()

In [ ]:
metric = MulticlassAUROC(num_classes=13, average=None, thresholds=10)
values = [ ]
for _ in range(10):
    mc_auroc = metric.forward(torch.randn(20, 13), torch.randint(13, (20,)))
    print(mc_auroc)
    values.append(mc_auroc)

metric.plot(values)
metric.plot([x[2] for x in values])
plt.show()

## Meshgrid

In [21]:
xs = torch.linspace(-5, 5, steps=11)
ys = torch.linspace(-5, 5, steps=11)
print(xs,ys)
grid_x, grid_y = torch.meshgrid(xs, ys, indexing='ij')

tensor([-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.]) tensor([-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.])


In [23]:
grid_x

tensor([[-5., -5., -5., -5., -5., -5., -5., -5., -5., -5., -5.],
        [-4., -4., -4., -4., -4., -4., -4., -4., -4., -4., -4.],
        [-3., -3., -3., -3., -3., -3., -3., -3., -3., -3., -3.],
        [-2., -2., -2., -2., -2., -2., -2., -2., -2., -2., -2.],
        [-1., -1., -1., -1., -1., -1., -1., -1., -1., -1., -1.],
        [ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.],
        [ 2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.],
        [ 3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.],
        [ 4.,  4.,  4.,  4.,  4.,  4.,  4.,  4.,  4.,  4.,  4.],
        [ 5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.]])

In [24]:
grid_y

tensor([[-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.]])

In [25]:
xs = torch.linspace(-5, 5, steps=11)
ys = torch.linspace(-5, 5, steps=11)
print(xs,ys)
grid_x, grid_y = torch.meshgrid(xs, ys, indexing='xy')

tensor([-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.]) tensor([-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.])


In [26]:
grid_x

tensor([[-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.]])

In [24]:
grid_y

tensor([[-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.],
        [-5., -4., -3., -2., -1.,  0.,  1.,  2.,  3.,  4.,  5.]])

In [ ]:
xs = torch.linspace(10, 20, steps=11)
ys = torch.linspace(0, 10, steps=11)
print(xs,ys)
grid_x, grid_y = torch.meshgrid(xs, ys, indexing='ij')

In [15]:
grid_x

tensor([[10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.],
        [11., 11., 11., 11., 11., 11., 11., 11., 11., 11., 11.],
        [12., 12., 12., 12., 12., 12., 12., 12., 12., 12., 12.],
        [13., 13., 13., 13., 13., 13., 13., 13., 13., 13., 13.],
        [14., 14., 14., 14., 14., 14., 14., 14., 14., 14., 14.],
        [15., 15., 15., 15., 15., 15., 15., 15., 15., 15., 15.],
        [16., 16., 16., 16., 16., 16., 16., 16., 16., 16., 16.],
        [17., 17., 17., 17., 17., 17., 17., 17., 17., 17., 17.],
        [18., 18., 18., 18., 18., 18., 18., 18., 18., 18., 18.],
        [19., 19., 19., 19., 19., 19., 19., 19., 19., 19., 19.],
        [20., 20., 20., 20., 20., 20., 20., 20., 20., 20., 20.]])

In [16]:
grid_y

tensor([[ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.],
        [ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.]])

In [20]:
xs = torch.linspace(10, 20, steps=11)
ys = torch.linspace(0, 10, steps=11)
print(xs,ys)
grid_x, grid_y = torch.meshgrid(xs, ys, indexing='xy')

tensor([10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.]) tensor([ 0.,  1.,  2.,  3.,  4.,  5.,  6.,  7.,  8.,  9., 10.])


In [18]:
grid_x

tensor([[10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.],
        [10., 11., 12., 13., 14., 15., 16., 17., 18., 19., 20.]])

In [19]:
grid_y

tensor([[ 0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.,  0.],
        [ 1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.,  1.],
        [ 2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.,  2.],
        [ 3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.,  3.],
        [ 4.,  4.,  4.,  4.,  4.,  4.,  4.,  4.,  4.,  4.,  4.],
        [ 5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.,  5.],
        [ 6.,  6.,  6.,  6.,  6.,  6.,  6.,  6.,  6.,  6.,  6.],
        [ 7.,  7.,  7.,  7.,  7.,  7.,  7.,  7.,  7.,  7.,  7.],
        [ 8.,  8.,  8.,  8.,  8.,  8.,  8.,  8.,  8.,  8.,  8.],
        [ 9.,  9.,  9.,  9.,  9.,  9.,  9.,  9.,  9.,  9.,  9.],
        [10., 10., 10., 10., 10., 10., 10., 10., 10., 10., 10.]])

## Decorator for model layer initializer

In [ ]:
def init_kaiming_experimental(layer, distribution = "kaiming_uniform", mode = "fan_in"):
    # layer_list = list(m.named_modules())
    # The code is creating a list called `layer_list` that contains all the named modules within the
    # object `m`.
    if distribution == "kaiming_uniform":   
        init_func = init.kaiming_uniform_
    else:
        init_func = init.kaiming_normal_
    # print(len(layer_list))

    # The code is iterating over a list of tuples `layer_list`, where each tuple contains a layer name
    # and a layer object. Inside the loop, the code unpacks each tuple into `layer_name` and `layer`
    # variables, allowing you to access the layer name and layer object separately for further
    # processing or manipulation.
    # for (layer_name, layer) in layer_list:
    print(f" layer :{layer} -- {type(layer)}")
    if isinstance(layer, nn.modules.Conv2d):
        print(f" {distribution} - Initialize Convolutional layer: weight shape: {layer.weight.shape}  ")
        init_func(layer.weight, mode= mode, nonlinearity='relu') # Example: Kaiming for ReLU
        if layer.bias is not None:
            print(" Initialize Bias")
            init.zeros_(layer.bias) # Biases are typically set to zero    
        print(" layer initialized ")
    
    elif isinstance(layer, nn.modules.Linear):
        print(f" Initialize linear layer: {layer.weight.shape}   {layer.bias.shape}")
        init_func(layer.weight, mode= mode, nonlinearity='relu') # Example: Kaiming for ReLU     
        if layer.bias is not None :
            print(" Initialize Bias")
            init.zeros_(layer.bias) # Biases are typically set to zero    
        print(" layer initialized ")
    else:
        print(f" layer is {type(layer)} - not initialized ")    


In [ ]:
def display(my_func, *args, **kwargs):

    #Iterate over all args, convert them to str, and join them
    args_str = ','.join(map(str,args))

    #Iterater over all kwargs, convert them into k=v and join them
    kwargs_str = ','.join('{}={}'.format(k,v) for k,v in kwargs.items())

    #Or using f-strings
    #kwargs_str = ','.join(f'{k}={v}' for k,v in kwargs.items()

    #Form the final representation by adding func name
    return "Call to {}({})".format(my_func.__name__, ','.join([args_str,kwargs_str]))

    #Or using f-strings
    #return f"processing {my_func.__name__}({','.join([args_str,kwargs_str])})"

In [ ]:
# def my_decorator(function, train_config):
#     def wrapper(*args, **kwargs):
#         args_str = ','.join(map(str,args))
#         kwargs_str = ','.join('{}={}'.format(k,v) for k,v in kwargs.items())
#         print("Call to {}({})".format(function.__name__, ','.join([args_str,kwargs_str])))
#         print(" Before the function call. - distribution : {}".format(train_config.initialization))
#         print()
#         result = function(*args, **kwargs)
#         print()
#         return result
#     return wrapper


# init_func = my_decorator(init_kaiming, train_config)

In [ ]:
# model.apply(init_kaiming(m, distribution=train_config.initialization))

## TQDM

## setup

In [ ]:
data_root ='./images'

In [ ]:
# get label to species mapping
test_csv_path = os.path.join(data_root, 'test.csv')
test_df = pd.read_csv(label_csv_path, engine='python')
test_df.head()
test_df.size

In [ ]:
# get label to species mapping
train_csv_path = os.path.join(data_root, 'train.csv')
train_df = pd.read_csv(train_csv_path, engine='python')
train_df.info()
train_df.head()

In [ ]:
label_names = list(train_df['class'].unique())
label_names.sort()
name_to_label = {x:i for i,x in enumerate(label_names)}
label_to_name = {i:x for x,i in name_to_label.items()}

train_df['label']=train_df['class'].map(lambda x: name_to_label[x])
# train_df['label'] = train_df['class'].map(lambda x: self.name_to_label[x])
train_df['image_path']=train_df['id'].map(lambda x: f"{os.path.join(data_root,str(x))}.jpg")
# train_df['image_path'] = train_df['id'].map(lambda x: f"{os.path.join(data_root,str(x))}.jpg")        
print(name_to_label)
print()
print(label_to_name)

train_split, val_split, y_train, y_val = train_test_split(
         train_df, train_df['label'], test_size=0.15, 
         stratify=train_df['label'], random_state=42)

In [ ]:
all_vc = train_df['class'].value_counts()
train_vc = train_split['class'].value_counts()
val_vc = val_split['class'].value_counts()
# print(all_vc)
# print(vc.mean(), vc.std())
print()
print(train_vc)
print(train_vc.mean(), train_vc.std())
# print(val_vc)
# print(val_vc.mean(), val_vc.std())

## Weighted Random Sampler for imbalanced datasets

In [ ]:
from collections import Counter
 
class_counts = Counter(train_split['label'])
len_train_split = class_counts.total()
class_weights = torch.Tensor([1.0/class_counts[x] for x in sorted(class_counts)])
sample_weights =[class_weights[x] for x in train_split.label]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len_train_split,
    replacement=True
)
print(f"assert sample_weights == sample_weights_2: {sample_weights == sample_weights_2}")

# train_labels = [x for x in train_split.label]
# sample_weights_2 = [class_weights[x] for x in train_labels]

class_counts_ordered = {x:class_counts[x] for x in sorted(class_counts) }
class_weights_dict = {x:1.0/class_counts[x] for x in class_counts_ordered.keys()}

print(class_counts.total())
print(class_counts_ordered)
print()
print(class_weights_dict, type(class_weights_dict[0]))
print()
print(class_weights)
print(class_weights.dtype)

# ttl = 0.0
# for i,j in class_weights.items():
#     print(f" {i:3d} {j:.6f}")
#     ttl += j
# print(ttl)
compare_list = []
for i, (x,y) in enumerate(zip(train_labels, sample_weights)):
    eq_test = np.equal(y.numpy(), class_weights_dict[x])
    compare_list.append(eq_test)
    if not eq_test:
        print(f" {x:3d}   {y.item():24.24f}  {y.numpy():24.24f}   Equal: {np.equal(y.numpy(), class_weights_dict[x])}")

print(np.all(compare_list))
len(train_split)

In [ ]:
print(sampler)
# np.all(compare_list)

## Mean and Stanard Deviation

**Mean and Standard Deviation of Images**
     
     length widths: 6536  min:320  max: 1919
     length heights: 6536  min:168  max: 1800
     ttl_images       :  6536  mean: tensor([0.6214, 0.4015, 0.3718]) std : tensor([0.2992, 0.2928, 0.2423])
     training_images  :  5555  mean: tensor([0.6214, 0.4015, 0.3718]) std : tensor([0.2992, 0.2928, 0.2423])
     validation_images:   981  mean: tensor([0.6215, 0.4015, 0.3718]) std : tensor([0.2991, 0.2927, 0.2424])

In [ ]:
widths = []
heights = []
for a,(b,c,d,e) in train_df.iterrows():
    print(f"{a} :  {b}  - {c} - {d} - {e}")
    # width, height = Image.open(b.image_path).convert("RGB").size
    # widths.append(width)
    # heights.append(height)
    break
# print(f" length widths: {len(widths)}  min:{min(widths)}  max: {max(widths)}")
# print(f" length heights: {len(heights)}  min:{min(heights)}  max: {max(heights)}")

In [ ]:
# preprocess = transforms.Compose([
#     transforms.Resize(512),
#     transforms.CenterCrop(512),
#     transforms.ToTensor()
#     ])

In [ ]:
def get_mean_std(data_df):
    batch_mean = torch.zeros(3)
    batch_mean_sqrd = torch.zeros(3)
    
    for counter, (idx, (_,_,_, filepath)) in enumerate(data_df.iterrows()):
        if (counter%100) == 0:
            print(f" counter: {counter} idx: {idx}   batch_mean: {batch_mean}   batch_mean^2 : {batch_mean_sqrd}")
        
        image = Image.open(train_df.iloc[0].image_path).convert("RGB")
        image = preprocess(image)   
        # loader = data_loader(data_root, transform)
    
        batch_mean += image.mean(dim=(1, 2)) # E[batch_i] 
        batch_mean_sqrd += (image ** 2).mean(dim=(1, 2)) #  E[batch_i**2]

    ttl_images = counter + 1
    print(f" ttl_images: {ttl_images}")
    
    mean = batch_mean / ttl_images
    
    # var[X] = E[X**2] - E[X]**2
    # E[X**2] = E[E[batch_1**2], E[batch_2**2], ...]
    # E[X]**2 = E[E[batch_1], E[batch_2], ...] ** 2

    var = (batch_mean_sqrd / ttl_images) - (mean ** 2)
    std = var ** 0.5
    
    # print('mean: {}, std: {}'.format(mean, std))
    return mean, std

In [ ]:
# m,s = get_mean_std(X_val)
# print(f" Final  mean: {m} std : {s}")

## Stratified Data Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
         train_df, train_df['label'], test_size=0.15, 
         stratify=train_df['label'], random_state=42)

In [ ]:
print(len(y_train))
y_train_len = len(y_train)
y_train_value_counts = y_train.value_counts()
for a,b in y_train_value_counts.items():
    print(f" class: {a:4d}   count: {b:6d}  pct: {b*100/y_train_len:.2f} " )

In [ ]:
print(len(y_val))
y_val_len = len(y_val)
y_val_value_counts = y_val.value_counts()
for a,b in y_val_value_counts.items():
    print(f" class: {a:4d}   count: {b:6d}  pct: {b*100/y_val_len:.2f} " )

In [ ]:
# print(X_train.head(10))
# print()
# print(y_train.head(10))

# print(X_test.head(10))
# print()
# print(y_test.head(10))

## Display an image

In [ ]:
idx = 3
image = Image.open(train_df.iloc[idx].image_path).convert("RGB")
label = train_df.iloc[idx]['class']

print(type(image), image.size)

In [ ]:
image = preprocess(image)
print(type(image), image.size())

In [ ]:
display_image(image, label)

# Dataset and Dataloader

## Test Dataset and Data Module

In [ ]:
# test_dataset =  KenyanFood13Dataset(data_root, data = 'test', split=None, image_shape=512, transform=preprocess)

# dataloader with dataset
# test_loader = DataLoader(test_dataset, batch_size=15, shuffle=False, num_workers=1)

In [ ]:
# iter_loader = iter(test_loader)

# images, labels = next(iter_loader)
# display_images(images, lbl_nms)

In [ ]:
# mean = torch.tensor([0.6214, 0.4015, 0.3718])
# std  = torch.tensor([0.2992, 0.2928, 0.2423])

mean = torch.tensor([0.485, 0.456, 0.406])
std  = torch.tensor([0.229, 0.224, 0.225])
normalize_transform =  transforms.Normalize(mean, std)

In [ ]:
# init the data module
data_module = KenyanFoodDataModule(
    data_root = './images', 
    batch_size = 12, 
    num_workers = 2, 
    image_shape = (224,224)
)
data_module.prepare_data()
data_module.setup()

In [ ]:
# len(data_module.train_dataset.data_dict)
# len(data_module.val_dataset.data_dict)
# len(data_module.test_dataset.data_dict)
# data_module.label_to_name

In [ ]:
iter_trn_loader = iter(data_module.train_dataloader())
iter_val_loader = iter(data_module.val_dataloader())
iter_tst_loader = iter(data_module.test_dataloader())

In [ ]:
images, labels = next(iter_trn_loader)

In [ ]:
print(labels)
names = [data_module.label_to_name[x] for x in labels.numpy()]
print(names)

In [ ]:
# display_images(images, labels, names)
# display_images(normalize_transform(images), labels)

In [ ]:
# v_images, v_labels = next(iter_val_loader)
# print(v_labels)
# v_names = [data_module.label_to_name[x] for x in v_labels.numpy()]
# print(v_names)

## Other Datasets

#### MonkeySpecies10Dataset

In [ ]:
class MonkeySpecies10Dataset(Dataset):
    """
    This custom dataset class takes root directory and train flag, 
    and returns dataset training dataset if train flag is true 
    else it returns validation dataset.
    """
    
    def __init__(self, data_root, train=True, image_shape=None, transform=None):
        
        """
        init method of the class.
        
         Parameters:
         
         data_root (string): path of root directory.
         
         train (boolean): True for training dataset and False for test dataset.
         
         image_shape (int or tuple or list): [optional] int or tuple or list. Defaut is None. 
                                             If it is not None image will resize to the given shape.
                                 
         transform (method): method that will take PIL image and transform it.
         
        """
        
        # get label to species mapping
        label_csv_path = os.path.join(data_root, 'monkey_labels.txt')
        
        self.label_df = pd.read_csv(label_csv_path, delimiter=' *, *', engine='python')
        
        
        # set image_resize attribute
        if image_shape is not None:
            if isinstance(image_shape, int):
                self.image_shape = (image_shape, image_shape)
            
            elif isinstance(image_shape, tuple) or isinstance(image_shape, list):
                assert len(image_shape) == 1 or len(image_shape) == 2, 'Invalid image_shape tuple size'
                if len(image_shape) == 1:
                    self.image_shape = (image_shape[0], image_shape[0])
                else:
                    self.image_shape = image_shape
            else:
                raise NotImplementedError 
                
        else:
            self.image_shape = image_shape
            
        # set transform attribute
        self.transform = transform
                
        num_classes = 10
        
        # initialize the data dictionary
        self.training_data = {
            'image_path': [],
            'label': []
        }
        
        # training data path, this will be used as data root if train = True
        if train:
            img_dir = os.path.join(data_root, 'training', 'training')
            
        # validation data path, this will be used as data root if train = False
        else:
            img_dir = os.path.join(data_root, 'validation', 'validation')
            
        for i in range(num_classes):
            class_path = os.path.join(img_dir, 'n{}'.format(i))
            for img in os.listdir(class_path):
                if img.endswith(".jpg") or img.endswith(".png"):
                    img_path = os.path.join(class_path, img)
                    self.training_data['image_path'].append(img_path)
                    self.training_data['label'].append(i)
                    
    def __len__(self):
        """
        return length of the dataset
        """
        return len(self.training_data['label'])
    
    
    def __getitem__(self, idx):
        """
        For given index, return images with resize and preprocessing.
        """
        
        image = Image.open(self.training_data['image_path'][idx]).convert("RGB")
        
        if self.image_shape is not None:
            image = F.resize(image, self.image_shape)
            
        if self.transform is not None:
            image = self.transform(image)
            
        target = self.training_data['label'][idx]
        
        return image, target            
                
        
    def common_name(self, label):
        """
        class label to common name mapping
        """
        return self.label_df['Common Name'][label]
    
    def latin_name(self, label):
        """
        class label to latin name mapping
        """
        return self.label_df['Latin Name'][label]
        
        
        

#### CatDogPandaDataModule

In [ ]:
class CatDogPandaDataModule(L.LightningDataModule):
    def __init__(self, batch_size, num_workers):

        super().__init__()

        self.batch_size = batch_size
        self.num_workers = num_workers

        mean = [0.485, 0.456, 0.406]
        std = [0.229, 0.224, 0.225]

        preprocess = transforms.Compose(
            [transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor()]
        )

        self.common_transforms = transforms.Compose(
            [preprocess, transforms.Normalize(mean, std)]
        )

        self.aug_transforms = transforms.Compose(
            [
                transforms.RandomResizedCrop(256),
                transforms.ColorJitter(
                    brightness=0.5, contrast=0.5, saturation=0.5, hue=0.3
                ),
                transforms.RandomHorizontalFlip(),
                transforms.RandomVerticalFlip(),
                transforms.RandomRotation(90),
                transforms.RandomGrayscale(p=0.1),
                self.common_transforms,
                transforms.RandomErasing(),
            ]
        )

    def prepare_data(self):

        curr_dir = os.getcwd()
        print("Preparing Cat, Dog, Panda dataset")
        # url = (
        #     "https://www.dropbox.com/sh/n5nya3g3airlub6/AACi7vaUjdTA0t2j_iKWgp4Ra?dl=1"
        # )
        # filename = os.path.join(curr_dir, r"animal-data.zip")
        # root = curr_dir

        # torchvision.datasets.utils.download_url(url, root, filename)

        # with zipfile.ZipFile(filename, "r") as f:

        #     # list the folders present in the zipfile
        #     directories = [info.filename for info in f.infolist() if info.is_dir()]

        #     # index 1 contatins the name of the root directory we are interested in
        #     # ["/", "cat-dog-panda/", "cat-dog-panda/training", ...]
        #     self.data_root = os.path.join(curr_dir, directories[1])

        #     # if data has not been extracted already (useful when experimenting again)
        #     # avoids extracting if dataset already extracted before
        #     if not os.path.isdir(self.data_root):
        #         # extract the zipfile contents
        #         f.extractall(curr_dir)
        self.data_root = "../../week5-project1/cat-dog-panda/"
        print("Preparation completed.")

    def setup(self, stage=None):
        train_data_path = os.path.join(self.data_root, "training")
        val_data_path = os.path.join(self.data_root, "validation")

        self.train_dataset = datasets.ImageFolder(
            root=train_data_path, transform=self.aug_transforms
        )
        self.val_dataset = datasets.ImageFolder(
            root=val_data_path, transform=self.common_transforms
        )

    def train_dataloader(self):
        # train loader
        train_loader = torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )
        return train_loader

    def val_dataloader(self):
        # validation loader
        test_loader = torch.utils.data.DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )
        return test_loader

#### MNISTDataModule

In [ ]:
class MNISTDataModule(L.LightningDataModule):
    def __init__(self, data_root, batch_size, num_workers):

        super().__init__()

        self.data_root = data_root
        self.batch_size = batch_size
        self.num_workers = num_workers

        self.train_test_transforms = transforms.Compose(
            [
                # Resize to 32X32
                transforms.Resize((32, 32)),
                # this re-scales image tensor values between 0-1. image_tensor /= 255
                transforms.ToTensor(),
                # subtract mean (0.1307) and divide by variance (0.3081).
                # This mean and variance are calculated on training data (verify for yourself)
                transforms.Normalize((0.1307,), (0.3081,)),
            ]
        )

    def prepare_data(self):
        # download
        datasets.MNIST(self.data_root, train=True, download=True)
        datasets.MNIST(self.data_root, train=False, download=True)

    def setup(self, stage=None):
        # create splits
        self.train_dataset = datasets.MNIST(
            self.data_root, train=True, transform=self.train_test_transforms
        )
        self.val_dataset = datasets.MNIST(
            self.data_root, train=False, transform=self.train_test_transforms
        )

    def train_dataloader(self):
        # train loader
        train_loader = torch.utils.data.DataLoader(
            self.train_dataset,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=self.num_workers,
        )
        return train_loader

    def val_dataloader(self):
        # validation loader
        test_loader = torch.utils.data.DataLoader(
            self.val_dataset,
            batch_size=self.batch_size,
            shuffle=False,
            num_workers=self.num_workers,
        )
        return test_loader

**Predefined methods and their usage:**


- **`prepare_data`:** This method is used for data preparation like  downloading and one-time preprocessing with the dataset. When  training on a distributed GPU, this will be called from a single GPU. So do not update or initiate attributes in this method. (Note: Do not do - `self.x = y`).


- **`setup`:** When you want to perform data operations on every GPU, this method is apt for it will call from every GPU. For example, perform train/val/test splits. 


- **`train_dataloader`:** This method returns dataloader with train dataset. 


- **`val_dataloader(s)`:** This method returns dataloader(s) with the validation dataset.


- **`test_dataloader(s)`:** This method returns dataloader(s) with the test dataset.


Get details of `LightningDataModule`<a href="https://pytorch-lightning.readthedocs.io/en/stable/extensions/datamodules.html" target="_blank">here</a>.

# <font style="color:green">Configuration [5 Points]</font></h1>

In [ ]:
@dataclass
class SystemConfiguration:
    '''
    Describes the common system setting needed for reproducible training
    '''
    seed: int = 21  # seed number to set the state of all random number generators
    cudnn_benchmark_enabled: bool = True  # enable CuDNN benchmark for the sake of performance
    cudnn_deterministic: bool = True  # make cudnn deterministic (reproducible training)

In [ ]:
@dataclass
class TrainingConfiguration:
    '''
    Describes configuration of the training process
    '''
    batch_size: int = 4      # 10 
    epochs: int = 20    # 50  
    learning_rate: float = 0.001  ## 0.1  # initial learning rate for lr scheduler
    log_interval: int = 5  
    test_interval: int = 1  
    data_root: str = "./images" 
    num_workers: int = 2  
    device: str = 'cuda'  
    model_name:str = 'unknown'
    subset_size: float = 1.00
    dropout_rate:float = 0.0
    run_name: str =""

In [ ]:
@dataclass
class Metrics():
    '''
    Metrics used in the training process
    '''
    best_loss_ep = -1
    best_loss = torch.tensor(np.inf)
    best_loss_acc = 0.0
    best_acc = 0.0

    # epoch train/test loss
    train_loss = np.array([])
    val_loss = np.array([])
    test_loss = np.array([])

    # epch train/test accuracy
    train_acc  = np.array([])
    val_acc  = np.array([]) 
    test_acc  = np.array([]) 

# <font style="color:green"> Train and Validation</font>


**Write the methods or classes to be used for training and validation.**

In [ ]:
try:
    del model
    print('model')
    del metrics
    print('metrics')
    del optimizer, scheduler
    print('optimizer, scheduler')
    del train_config, sys_config
    print('train_config, sys_config')
    del train_loader, test_loader
    print('train_loader, test_loader')
except Exception as e:
    print(e)
    pass

gc.collect()
torch.cuda.empty_cache()

In [ ]:
# MODEL = globals()[MODEL_NAME]
model = MODEL(p=DROPOUT_RATE)
init_model(model)
print(f"{model.name}")
print(model)

metrics = Metrics()
sys_config = SystemConfiguration()
train_config = TrainingConfiguration()

train_config.batch_size: int = BATCH_SIZE  # 10 
train_config.epochs: int = EPOCHS   # 50  
train_config.learning_rate: float = LEARNING_RATE  ## 0.1  # initial learning rate for lr scheduler 
train_config.model_name = MODEL_NAME
train_config.dropout_rate = DROPOUT_RATE
train_config.subset_size = SUBSET_SIZE

do = f"_p{train_config.dropout_rate:3.1f}" if train_config.dropout_rate != 0.0 else ""
train_config.run_name = f"{MODEL_NAME}_BS_{BATCH_SIZE:03d}_LR_{LEARNING_RATE:.1e}{do}_n"

print(sys_config)
print(train_config)

# data loader
train_loader, test_loader = get_data(batch_size=train_config.batch_size,
                                     data_root=train_config.data_root,
                                     num_workers=train_config.num_workers,
                                     data_augmentation=True,
                                     subset_size=train_config.subset_size
                                     )

# optimizer
optimizer = optim.Adam(model.parameters(),
                       lr = train_config.learning_rate
                      )

scheduler  = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, 
                                                  patience=10, cooldown=5, 
                                                  threshold=0.00001, threshold_mode='rel', 
                                                  min_lr=0, eps=1e-08)

wandb_data = {'project_name': 'OpenCV_Pytorch',
              'exp_id' : None,
              'exp_name': train_config.run_name,
              'exp_description': train_config.run_name
             }

print(f" wandb data        : {wandb_data}\n\n")

print(f" Model name        : {train_config.model_name}   \n"
      f" epochs            : {train_config.epochs} \n"
      f" Batch size        : {train_config.batch_size}   \n"
      f" Initial LR        : {train_config.learning_rate} \n"
      f" Data Subset Ratio : {train_config.subset_size}   \n"
      f" Dropout rate      : {train_config.dropout_rate}  \n"
      f" Current LR        : {optimizer.param_groups[0]['lr']:.2e}   {scheduler.get_last_lr()[0]:.2e} \n"
      f" Scheduler Best    : {scheduler.best:f}   {scheduler} \n\n"
      f" Run Name          : {train_config.run_name}")
      # f" last epoch  : {epochs}        Next run - from epoch {beginning_epoch} to {ending_epoch-1}\n\n"
      # f" Best loss   : {best_loss:.4f}    Best Accuracy: {best_accuracy:.4f} \n\n"
      # f" len train_loss: {len(epoch_train_loss)}  len  test_loss : {len(epoch_test_loss)}")
    

In [ ]:
# wandb_session = wandb_init(SimpleNamespace(**wandb_data))

In [ ]:
# model, metrics = main(model, 
#                       optimizer=optimizer, 
#                       scheduler=scheduler, 
#                       system_configuration=sys_config,
#                       train_config=train_config, 
#                       metrics = metrics,
#                       train_loader = train_loader, 
#                       test_loader = test_loader,
#                       wandb_session = wandb_session
#                     )

# <font style="color:green">Models</font>

In [ ]:
class  pretrained_resnet50(nn.Module):
    def __init__(self, transfer_learning=True, num_target_classes=13):
        super().__init__()
        # init a pretrained resnet
        backbone = models.resnet50(weights="DEFAULT")
        num_filters = backbone.fc.in_features
        ## Extract all layers prior to the fully connected layer
        layers = list(backbone.children())[:-1]   
        ## call those layers "feature_extractor"
        self.feature_extractor = nn.Sequential(*layers)
        self.feature_extractor.eval()
        # use the pretrained model to classify cifar-10 (10 image classes)
         
        self.classifier = nn.Linear(num_filters, num_target_classes)

    def forward(self, x):
        with torch.no_grad():
            representations = self.feature_extractor(x).flatten(1)
 
        x = self.classifier(representations)
        return x

In [ ]:
def pretrained_resnet18(nn.Module):
    def __init__(self, transfer_learning=True, num_class=3):
        
        self.resnet = models.resnet18(weights='DEFAULT')
        
        if transfer_learning:
            for param in self.resnet.parameters():
                param.requires_grad = False
                
        last_layer_in = resnet.fc.in_features
        self.resnet.fc = nn.Linear(last_layer_in, num_class)
        
    def forward(self, x):
        return self.resnet(x)

In [ ]:
# vgg16 = models.vgg16(pretrained=True)

# print_model_summary(vgg16, image_size=(3,512,512))

# print(vgg16)

# vgg16.classifier

# backbone = list(vgg16.features.children())
# backbone.append(vgg16.avgpool)


# for i, l in enumerate(backbone):
#     print(f"layer {i:3d} :  {l}")

In [ ]:
def pretrained_resnet18(transfer_learning=True, num_class=13):
    # resnet = models.resnet18(weights='DEFAULT')
    resnet = models.resnet18()
    
    if transfer_learning:
        for param in resnet.parameters():
            param.requires_grad = False
            
    last_layer_in = resnet.fc.in_features
    resnet.fc = nn.Linear(last_layer_in, num_class)
    
    return resnet

In [ ]:
class  pretrained_resnet50(nn.Module):
    def __init__(self, transfer_learning=True, num_target_classes=13):
        super().__init__()
        # init a pretrained resnet
        backbone = models.resnet50(weights="DEFAULT")
        num_filters = backbone.fc.in_features
        ## Extract all layers prior to the fully connected layer
        layers = list(backbone.children())[:-1]   
        ## call those layers "feature_extractor"
        self.feature_extractor = nn.Sequential(*layers)
        self.feature_extractor.eval()
        # use the pretrained model to classify cifar-10 (10 image classes)
         
        self.classifier = nn.Linear(num_filters, num_target_classes)

    def forward(self, x):
        with torch.no_grad():
            representations = self.feature_extractor(x).flatten(1)
 
        x = self.classifier(representations)
        return x

In [ ]:
class  pretrained_resnet50(nn.Module):
    
    def __init__(self,num_classes=13, 
                 finetune = ['fc'],
                 weights=models.ResNet50_Weights.IMAGENET1K_V2):
    
        super().__init__()
        self.name = "resnet50"
        self.finetune =finetune        
        self.tuning_layers = ''.join(['fc']+sorted([x[-1] for x in self.finetune if (x[:-1]=='layer' and x != 'fc')], reverse=True))
        
        # init a pretrained resnet
        self.resnet = models.resnet50(weights=weights)
        last_layer_in = self.resnet.fc.in_features
        self.resnet.fc = nn.Linear(last_layer_in, last_layer_in//4)
        self.resnet.fc2 = nn.Linear(last_layer_in //4, num_classes)
        
        for param in self.resnet.parameters():
            param.requires_grad = False
        
        for ft_layer in finetune:
            print(f" turn on gradients for layer {ft_layer}")
            layer = getattr(self.resnet, ft_layer)
            print(layer)
            print()

            for param in layer.named_parameters():
                print(f" param : {param[0]:35s}  Shape: {param[1].shape}  Requires Grad: {param[1].requires_grad}",end ='')
                param[1].requires_grad = True
                print(f"  ---> {param[1].requires_grad}")
            
    def forward(self, x):
        x = self.resnet(x)
        return x


In [ ]:
resnet = pretrained_resnet50()
print()
print(resnet)   

In [ ]:
# print(resnet)
# print_model_summary(resnet)
# for param in resnet.parameters():
#      print(param.shape, param.requires_grad)

# resnet.fc.in_features
# resnet.layer1
# del resnet

In [ ]:
# getattr(models, 'resnet18')

In [ ]:
# layers = list(resnet50.children())
# for layer  in layers[:-1]:
#     print('\n'+'='*100)
#     print(layer)
#     print('='*100+'\n')
# print(layers[-1])

In [ ]:
data_module = KenyanFoodDataModule(
    data_root = './images', 
    batch_size = 48, 
    num_workers = 4, 
    image_shape = (224,224)
)

# PyTorch Lightning

In [ ]:
class ImagenetTransferLearning(L.LightningModule):
    def __init__(self):
        super().__init__()

        # init a pretrained resnet
        backbone = models.resnet50(weights="DEFAULT")
        num_filters = backbone.fc.in_features
        ## Extract all layers prior to the fully connected layer
        layers = list(backbone.children())[:-1]
        
        ## call those layers "feature_extractor"
        self.feature_extractor = nn.Sequential(*layers)
        self.feature_extractor.eval()

        # use the pretrained model to classify cifar-10 (10 image classes)
        num_target_classes = 10
        self.classifier = nn.Linear(num_filters, num_target_classes)

    def forward(self, x):
        with torch.no_grad():
            representations = self.feature_extractor(x).flatten(1)
 
        x = self.classifier(representations)
 

In [ ]:
class TransferLearner(L.LightningModule):
    def __init__(
        self,
        resnet_model_name="resnet18",
        weights=models.ResNet18_Weights.DEFAULT,
        fine_tune_start=99,
        learning_rate=0.001,
        num_classes=13,
    ):
        super().__init__()
        self.t_acc, self.t_loss = 0.0 , 0.0
        self.v_acc, self.t_loss = 0.0 , 0.0
        
        self.save_hyperparameters()
        # print(getattr(models, self.hparams.resnet_model_name)())
        # resnet = getattr(models, self.hparams.resnet_model_name)(
        #     weights=self.hparams.weights
        # )
        resnet = pretrained_resnet50()
        # if self.hparams.weights:
        #     print("self.hparams.weights")
        #     for param in resnet.parameters():
        #         param.requires_grad = False

        # if self.hparams.weights and self.hparams.fine_tune_start <= 1:
        #     for param in resnet.layer1.parameters():
        #         param.requires_grad = True

        # if self.hparams.weights and self.hparams.fine_tune_start <= 2:
        #     for param in resnet.layer2.parameters():
        #         param.requires_grad = True

        # if self.hparams.weights and self.hparams.fine_tune_start <= 3:
        #     for param in resnet.layer3.parameters():
        #         param.requires_grad = True

        # if self.hparams.weights and self.hparams.fine_tune_start <= 4:
        #     for param in resnet.layer4.parameters():
        #         param.requires_grad = True

        # last_layer_in = resnet.fc.in_features
        # resnet.fc = nn.Linear(last_layer_in, self.hparams.num_classes)

        self.resnet = resnet

        # Initializing the required metric objects.
        self.mean_train_loss = MeanMetric()
        self.mean_train_acc = MulticlassAccuracy(num_classes=self.hparams.num_classes, average="micro")
        self.mean_valid_loss = MeanMetric()
        self.mean_valid_acc = MulticlassAccuracy(num_classes=self.hparams.num_classes, average="micro")

    def forward(self, x):
        """
        Run data through the model 
        """
        return self.resnet(x)



    def training_step(self, batch, *args, **kwargs):
        """
        Params:

        batch – The output of your data iterable, normally a DataLoader.
        batch_idx – The index of this batch.
        dataloader_idx – The index of the dataloader that produced this batch. (only if multiple dataloaders used)
        """

        # get data and labels from batch
        data, target = batch

        # get prediction
        output = self(data)

        # # calculate batch loss
        loss = F.cross_entropy(output, target)

        # # Batch Predictions.
        pred_batch = output.detach().argmax(dim=1)

        self.t_loss = self.mean_train_loss(loss, weight=data.shape[0])
        self.t_acc = self.mean_train_acc(pred_batch, target)

        # Arguments such as on_epoch, on_step and logger are set automatically depending on
        # hook methods it's been called from
        # self.log("train/batch_loss", self.mean_train_loss, prog_bar=True, logger=True)

        # logging and adding current batch_acc to progress_bar
        # self.log("train/batch_acc", self.mean_train_acc, prog_bar=True, logger=True)

        # return loss
        pass

    
    def on_train_epoch_end(self):
        """Calculate epoch level metrics for the train set"""
        self.log("train/loss", self.mean_train_loss, prog_bar=True, logger=True)
        self.log("train/acc", self.mean_train_acc, prog_bar=True, logger=True)
        self.log("step", self.current_epoch, logger=True)
        pass
    
    def validation_step(self, batch, *args, **kwargs):

        # get data and labels from batch
        data, target = batch

        # get prediction
        output = self(data)

        # calculate loss
        loss = F.cross_entropy(output, target)

        # # Batch Predictions.
        pred_batch = output.argmax(dim=1)

        # # Update logs.
        self.v_loss = self.mean_valid_loss(loss, weight=data.shape[0])
        self.v_acc  = self.mean_valid_acc(pred_batch, target)

    def on_validation_epoch_end(self):
        # """Calculate epoch level metrics for the validation set"""
        print(f" step:  {self.current_epoch:4d}   train acc: {self.t_acc*100:.3f}  loss: {self.t_loss:.6f}"
              f"    Val acc: {self.v_acc*100:.3f}  loss: {self.v_loss:.6f}")
        self.log("valid/loss", self.mean_valid_loss, prog_bar=True, logger=True)
        self.log("valid/acc", self.mean_valid_acc, prog_bar=True, logger=True)
        self.log("step", self.current_epoch, logger=True)
        pass
        
    def configure_optimizers(self):
        return torch.optim.SGD(self.parameters(), lr=self.hparams.learning_rate)    

In [ ]:
model = TransferLearner(num_classes=13, 
                        fine_tune_start=4,
                        learning_rate=1.0e-04)

# <font style="color:green">Utils</font>

**Define those methods or classes, which have  not been covered in the above sections.**

# <font style="color:green">Experiment</font>

**Choose your optimizer and LR-scheduler and use the above methods and classes to train your model.**

In [ ]:
trainer = L.Trainer(
    max_epochs=10,
    accelerator='auto',
    devices=1,
    precision='32-true',  # Use mixed precision for faster training
    # callbacks=[
    #     L.pytorch.callbacks.ModelCheckpoint(
    #         monitor='val_loss',
    #         mode='min',
    #         save_top_k=3,
    #         filename='model-{epoch:02d}-{val_loss:.2f}'
    #     ),
    #     L.pytorch.callbacks.EarlyStopping(
    #         monitor='val_loss',
    #         patience=10,
    #         mode='min'
    #     )
    # ],
    logger=L.pytorch.loggers.TensorBoardLogger('logs/', name='image_classifier')
)

In [ ]:
trainer.fit(model, data_module)

# <font style="color:green">TensorBoard Log</font>

**Share your TensorBoard scalars logs here You can also share (not mandatory) your GitHub link, if you have pushed this project in GitHub.**

In [ ]:
print('done ')

# <font style="color:green">Kaggle Profile Link</font>

**Share your Kaggle profile link  with us here to score , points in  the competition.**

**For full points, you need a minimum accuracy of `75%` on the test data. If accuracy is less than `70%`, you gain  no points for this section.**

**Submit `submission.csv` (prediction for images in `test.csv`), in the `Submit Predictions` tab in Kaggle, to get evaluated for  this section.**